# Demand Forecasting in Production

Companion notebook for the [Demand Forecasting in Production lesson](https://ml-viz-ruby.vercel.app/courses/time-series/04-demand-forecasting-in-production).

We implement hierarchical bottom-up forecasting, Prophet-style additive decomposition with holiday effects, quantile regression for ETA prediction, and ensemble blending. Pure NumPy + minimal stdlib.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
rng = np.random.default_rng(13)

## 1 — Hierarchical bottom-up forecasting

We forecast 4 bottom-level series (stores) independently, then aggregate to get region and national forecasts.

In [ ]:
# Simulate 52 weeks of weekly demand for 4 stores
T = 52
store_demand = np.array([
    10 + 2*np.sin(2*np.pi*np.arange(T)/52) + rng.normal(0,1,T),  # store A1
    15 + 3*np.cos(2*np.pi*np.arange(T)/52) + rng.normal(0,1,T),  # store A2
    8  + 1.5*np.sin(2*np.pi*np.arange(T)/52+1) + rng.normal(0,1,T), # store B1
    20 + 4*np.cos(2*np.pi*np.arange(T)/52+2) + rng.normal(0,1,T),  # store B2
])

# Hierarchy: national = sum(all stores), region_A = store_A1+A2, region_B = B1+B2
national   = store_demand.sum(0)
region_a   = store_demand[:2].sum(0)
region_b   = store_demand[2:].sum(0)

# Simple forecast: last-4-weeks mean for each bottom-level series
def naive_forecast(series, h=4):
    return np.full(h, series[-4:].mean())

forecasts_stores = [naive_forecast(s) for s in store_demand]

# Bottom-up aggregation
fc_national = sum(forecasts_stores)
fc_region_a = forecasts_stores[0] + forecasts_stores[1]
fc_region_b = forecasts_stores[2] + forecasts_stores[3]

print(f"National forecast (4 weeks): {fc_national.round(1)}")
print(f"Region A:                    {fc_region_a.round(1)}")
print(f"Region B:                    {fc_region_b.round(1)}")
print(f"Sum check (A+B == National): {np.allclose(fc_region_a + fc_region_b, fc_national)}")

## 2 — Special event (holiday) effects

We model a Thanksgiving spike as an additive holiday effect — the same pattern Prophet uses.

In [ ]:
# Simulate demand with a Thanksgiving spike (week 47)
holiday_weeks = [47]     # Thanksgiving
holiday_effect = 20.0    # +20 units in holiday week

demand = national.copy()
for w in holiday_weeks:
    demand[w] += holiday_effect

# Simple holiday-aware forecast: naive + holiday bump for next period
def holiday_aware_forecast(series, holiday_weeks_future, holiday_effect, h=4):
    base = series[-4:].mean()
    forecast = np.full(h, base)
    # weeks 53-56 (next 4 weeks after week 52)
    for offset in range(h):
        week = 52 + offset
        if week in holiday_weeks_future:
            forecast[offset] += holiday_effect
    return forecast

fc_holiday = holiday_aware_forecast(demand, holiday_weeks_future=[52+0], holiday_effect=holiday_effect)
fc_naive   = naive_forecast(demand, h=4)

print(f"Naive forecast (no holiday): {fc_naive.round(1)}")
print(f"Holiday-aware forecast:      {fc_holiday.round(1)}")
print(f"Difference (holiday bump):   {(fc_holiday - fc_naive).round(1)}")

## 3 — Quantile regression for ETA prediction

We train two linear quantile regression models for the p25 and p75 delivery time quantiles, producing a confidence interval for the ETA.

In [ ]:
# Simulate delivery data: features (distance, traffic_index) -> delivery_time
n_samples = 500
distance    = rng.uniform(0.5, 10, n_samples)          # km
traffic_idx = rng.uniform(0.5, 3.0, n_samples)         # 1 = free flow, 3 = heavy
true_base   = 5 + 3*distance + 4*traffic_idx
noise       = rng.exponential(3, n_samples)            # right-skewed noise
delivery    = true_base + noise                        # minutes

X = np.column_stack([np.ones(n_samples), distance, traffic_idx])

def pinball_loss(q, y, y_hat):
    e = y - y_hat
    return np.where(e >= 0, q*e, (q-1)*e).mean()

def quantile_regression_gradient(X, y, q, n_iter=2000, lr=0.001):
    """Gradient descent for quantile regression (pinball loss)."""
    w = np.zeros(X.shape[1])
    for _ in range(n_iter):
        y_hat = X @ w
        e = y - y_hat
        grad = -X.T @ np.where(e >= 0, q, q-1) / len(y)
        w -= lr * grad
    return w

w_p25 = quantile_regression_gradient(X, delivery, q=0.25)
w_p75 = quantile_regression_gradient(X, delivery, q=0.75)

# Predict interval for new order: 3 km, moderate traffic (1.5)
x_new = np.array([1.0, 3.0, 1.5])
eta_p25 = x_new @ w_p25
eta_p75 = x_new @ w_p75
print(f"Estimated delivery time: {eta_p25:.0f}–{eta_p75:.0f} minutes")
print(f"True expected time: {5 + 3*3.0 + 4*1.5:.0f} min (without noise)")

## ✏️ Your turn

**Exercise.** Implement `walk_forward_mae(series, window, horizon, n_folds)`:
- For each fold, train on `series[:t]`, forecast `horizon` steps, compute MAE vs actual.
- Return the mean MAE across all `n_folds` evaluation windows.

This simulates the proper expanding-window validation used in production forecasting.

In [ ]:
def walk_forward_mae(series, window=20, horizon=4, n_folds=5):
    """
    series: 1D array of time series values
    window: minimum training window
    horizon: number of steps to forecast
    n_folds: number of evaluation windows
    Returns: mean MAE across folds
    """
    # TODO(you): for each fold, take a slice of the series, compute naive forecast,
    # and compute MAE against the actual next `horizon` values
    return ...

mae = walk_forward_mae(national, window=20, horizon=4, n_folds=5)
print(f"Walk-forward MAE: {mae:.2f}")

In [ ]:
# Assertion
def _ref_wf(series, window=20, horizon=4, n_folds=5):
    n = len(series)
    step = (n - window - horizon) // n_folds
    maes = []
    for i in range(n_folds):
        t = window + i * step
        pred = np.full(horizon, series[max(0,t-4):t].mean())
        actual = series[t:t+horizon]
        maes.append(np.abs(pred - actual).mean())
    return np.mean(maes)
ref = _ref_wf(national)
result = walk_forward_mae(national)
assert abs(result - ref) < 5.0, f"Expected ~{ref:.2f}, got {result:.2f}"
print(f"✓ walk_forward_mae correct (MAE ≈ {result:.2f})")

<details><summary>Solution</summary>

```python
def walk_forward_mae(series, window=20, horizon=4, n_folds=5):
    n = len(series)
    step = (n - window - horizon) // n_folds
    maes = []
    for i in range(n_folds):
        t = window + i * step
        train = series[:t]
        forecast = np.full(horizon, train[-4:].mean())
        actual = series[t:t+horizon]
        maes.append(np.abs(forecast - actual).mean())
    return np.mean(maes)
```
</details>